# Trading Simulation (No Training)

This notebook runs the same simulation logic as Step 5.1 in `training_workflow.ipynb` but allows arbitrary date ranges.
It loads features/labels, aligns and cleans data, loads a trained model, predicts, and then runs `simulate_trading`.


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from qate.core.model import Market, ExchangeName, Symbol
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange

from triple_barrier.triple_barrier import TripleBarrierConfig
from training import Dataset, LGBMTrainer
from simulation.v1 import simulate_trading_strict as simulate_trading, plot_simulation_results

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
# === Date range (editable) ===
DATA_START_DATE = '20250919'
DATA_END_DATE = '20250921'
DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)

# === Markets (same as training) ===
FEATURE_MARKET = Market(ExchangeName.BINANCE, Symbol.XRP_USDT)
LABEL_MARKET = Market(ExchangeName.GMO, Symbol.XRP_JPY)

# === Triple Barrier Config (must match training) ===
_BARRIER_MULTIPLIER = 1.0
_RETURN_SHIFT_SEC = 20
_TIMEOUT_SEC = 30
_MIN_VOL_SPREAD_RATIO = 1.0

_VOLATILITY_LOOKBACK_SEC = 900
_EMA_SPAN_SECONDS = 60
_VOL_RESAMPLE_INTERVAL = '1s'
_LABEL_RESAMPLE_INTERVAL = '100ms'

CONFIG = TripleBarrierConfig(
    barrier_multiplier=_BARRIER_MULTIPLIER,
    return_shift_sec=_RETURN_SHIFT_SEC,
    timeout_sec=_TIMEOUT_SEC,
    min_vol_spread_ratio=_MIN_VOL_SPREAD_RATIO,
    volatility_lookback_sec=_VOLATILITY_LOOKBACK_SEC,
    ema_span_sec=_EMA_SPAN_SECONDS,
    vol_resample_interval=_VOL_RESAMPLE_INTERVAL,
    label_resample_interval=_LABEL_RESAMPLE_INTERVAL,
)

TARGET_COLUMN = 'label'
MODEL_PATH = '../models/lgbm_model'

print(f'Feature market: {FEATURE_MARKET}')
print(f'Label market:   {LABEL_MARKET}')
print(f'Date range:     {DT_RANGE.start_str} -> {DT_RANGE.end_str}')
print(f'Label param_id: {CONFIG.param_id}')


In [ ]:
# Load features and labels
parquet_store = ParquetStore(sys_env.get_parquet_root_dir())
feature_store = parquet_store.create_feature(FEATURE_MARKET)
label_store = parquet_store.create_label('TripleBarrierLabel_tmp', LABEL_MARKET, CONFIG.param_id)

features_df = feature_store.read(DT_RANGE)
labels_df = label_store.read(DT_RANGE)

print(f'Features: {features_df.shape[0]:,} rows x {features_df.shape[1]} columns')
print(f'Labels:   {labels_df.shape[0]:,} rows x {labels_df.shape[1]} columns')

def prepare_df(df):
    """Ensure DatetimeIndex in UTC, handle ts column if present."""
    if 'ts' in df.columns:
        df.index = pd.to_datetime(df['ts'], unit='s')
        df = df.drop(columns=['ts'])
    return df.sort_index()

features_df = prepare_df(features_df)
labels_df = prepare_df(labels_df)

print(f'\nFeatures range: {features_df.index.min()} -> {features_df.index.max()}')
print(f'Labels range:   {labels_df.index.min()} -> {labels_df.index.max()}')

# Original label distribution
print('\nTriple-barrier label distribution:')
print(labels_df[TARGET_COLUMN].value_counts().sort_index().to_string())
print(f'  1 = Win, -1 = Loss, 0 = Timeout')

display(features_df.head())
display(labels_df.head())

In [ ]:
# Align, filter, and convert labels
dataset = Dataset(
    features_df=features_df,
    labels_df=labels_df,
    target_column=TARGET_COLUMN,
)

dataset.align(include_label_columns=['volatility_ema', 'spread_pct_ema'])
dataset.filter_by_vol_spread(min_ratio=CONFIG.min_vol_spread_ratio)
dataset.convert_to_multiclass()

print('After alignment and filtering:')
X_aligned, y_aligned = dataset.get_aligned_data()
print(f'  {len(X_aligned):,} rows x {X_aligned.shape[1]} features')
print(f'  Label distribution: {y_aligned.value_counts().sort_index().to_dict()}')

from training.plot_labels import plot_labels

plot_labels(y_aligned, CONFIG)

# Missing values
na_pct = (X_aligned.isna().sum() / len(X_aligned) * 100).sort_values(ascending=False)
na_nonzero = na_pct[na_pct > 0]

if len(na_nonzero) > 0:
    print(f'{len(na_nonzero)} columns with missing values (top 20):')
    print(na_nonzero.head(20).to_string())
else:
    print('No missing values.')


In [ ]:
# Load trained model and align feature columns
trainer = LGBMTrainer.load(MODEL_PATH)

print(f'\nLoaded model from {MODEL_PATH}')
# Force dataset to use model's feature set
# dataset.feature_columns = trainer.feature_names

print(f"features: {len(trainer.feature_names)}")
print(f"features: {trainer.feature_names}")
# Clean NA columns/values (same thresholds as training)
dataset.drop_na_features(threshold=0.5)
dataset.fill_na(method='ffill')

X, y = dataset.get_aligned_data()

print(f'\nAfter aligning to model features and cleaning NAs:')
print(f'  {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  Label distribution: {y.value_counts().sort_index().to_dict()}')

print("X.columns:")
print(X.columns.tolist())

# Ensure all model features exist
missing = [c for c in trainer.feature_names if c not in X.columns]
if missing:
    print(f'Warning: {len(missing)} features missing, filling with 0. Example: {missing[:5]}')
    for c in missing:
        X[c] = 0.0

# Order columns to match the model
X = X[trainer.feature_names]

print(f'Final feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns')

In [ ]:
# Predict (multiclass)
y_proba = trainer.predict_proba(X)
y_pred = np.argmax(y_proba, axis=1)

print('Prediction distribution:')
unique, counts = np.unique(y_pred, return_counts=True)
for cls, cnt in zip(unique, counts):
    name = {0: 'Short', 1: 'NoEntry', 2: 'Long'}.get(cls, str(cls))
    print(f'  {name} ({cls}): {cnt:,} ({cnt/len(y_pred):.1%})')


In [ ]:
# Run simulation (same method as training_workflow step 5.1)
sim_result = simulate_trading(
    predictions=y_pred,
    labels_df=labels_df,
    test_indices=X.index,
)

sim_result.print_summary()
plot_simulation_results(sim_result)
